# P03 — Análise Estatística de ERPs

**Objetivo:** Análise estatística rigorosa de ERPs usando:

1. **Cluster-based permutation test** (controle de múltiplas comparações espaço-temporais)
2. **Mixed-effects models** (equivalente ao `lme4` do R)
3. **Bootstrap** para ICs

**Pré-requisitos:** dados pré-processados de P03 (ou usar sintéticos abaixo)

In [ ]:
# Setup
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
from setup import ERP_COMPONENTS

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import mne
from mne.stats import (
    spatio_temporal_cluster_1samp_test,
    permutation_cluster_1samp_test,
    ttest_1samp_no_p,
)

from scipy import stats
from scipy.stats import ttest_rel, ttest_ind

np.random.seed(42)
print("✅ Setup completo")

In [ ]:
# 1. Simular dados de múltiplos sujeitos
print("=== Simulando dados de N=20 sujeitos ===\n")

n_subjects = 20
n_channels = 32
n_times = 301  # -200 a 1000 ms a 250 Hz
times = np.linspace(-0.2, 1.0, n_times)
sfreq = 250

ch_names = ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4',
            'O1', 'O2', 'F7', 'F8', 'T7', 'T8', 'P7', 'P8',
            'Fz', 'Cz', 'Pz', 'Oz', 'FC1', 'FC2', 'CP1', 'CP2',
            'AF3', 'AF4', 'PO3', 'PO4', 'F5', 'F6', 'C5', 'C6']

# Função para gerar dados com efeito
def generate_subject(condition, with_effect=True):
    """Gera dados de EEG para um sujeito, com ou sem efeito N170."""
    data = np.random.randn(n_channels, n_times) * 1e-6
    if with_effect:
        # Adicionar N170 em canais occipitais (130-210 ms)
        occipital = [ch_names.index(ch) for ch in ['O1', 'O2', 'P7', 'P8', 'Oz']]
        n170_idx = (times >= 0.13) & (times <= 0.21)
        for ch in occipital:
            # Forma gaussiana centrada em 170 ms
            t_peak = 0.170
            sigma = 0.025
            gaussian = np.exp(-0.5 * ((times - t_peak) / sigma) ** 2)
            # Condição 1 (palavras): N170 mais negativo
            # Condição 2 (pseudopalavras): N170 menos negativo
            amp = -5e-6 if condition == 'palavra' else -2e-6
            data[ch] += amp * gaussian
    return data

# Gerar 2 condições × 20 sujeitos
all_subjects = {}
for cond in ['palavra', 'pseudopalavra']:
    all_subjects[cond] = []
    for subj in range(n_subjects):
        all_subjects[cond].append(generate_subject(cond, with_effect=True))

print(f"✅ {n_subjects} sujeitos × 2 condições gerados")
print(f"   Shape: {all_subjects['palavra'][0].shape}")

In [ ]:
# 2. Calcular grand average
print("\n=== Calculando grand average ===\n")

evokeds = {}
for cond in ['palavra', 'pseudopalavra']:
    # Stack sujeitos: (n_subjects, n_channels, n_times)
    data = np.stack(all_subjects[cond])
    evokeds[cond] = {
        'mean': data.mean(axis=0),
        'sem': data.std(axis=0) / np.sqrt(n_subjects),
        'n': n_subjects
    }
    print(f"  {cond}: n={evokeds[cond]['n']}, mean shape={evokeds[cond]['mean'].shape}")

In [ ]:
# 3. Plot ERPs com banda de erro
print("\n=== Plotando ERPs ===")

ch = 'O1'
ch_idx = ch_names.index(ch)

fig, ax = plt.subplots(figsize=(12, 6))
colors = {'palavra': '#1f77b4', 'pseudopalavra': '#ff7f0e'}

for cond, color in colors.items():
    mean = evokeds[cond]['mean'][ch_idx] * 1e6  # µV
    sem = evokeds[cond]['sem'][ch_idx] * 1e6
    ax.plot(times * 1000, mean, label=cond, color=color, linewidth=2)
    ax.fill_between(times * 1000, mean - sem, mean + sem,
                    alpha=0.3, color=color)

# Sombrear N170
ax.axvspan(130, 210, alpha=0.15, color='gray', label='N170 (130-210 ms)')
ax.axhline(0, color='black', linestyle='--', linewidth=0.5)
ax.axvline(0, color='red', linestyle='--', linewidth=0.5, alpha=0.5)
ax.set_xlim(-200, 600)
ax.set_xlabel('Tempo (ms)')
ax.set_ylabel('Amplitude (µV)')
ax.set_title(f'Grand Average ERP — {ch} (N=20)')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('10_grand_average_n170.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Salvo: 10_grand_average_n170.png")

In [ ]:
# 4. Cluster-based permutation test
print("\n=== Cluster-based Permutation Test ===\n")
print("Testando palavra vs. pseudopalavra...\n")

# Diferença: palavra - pseudopalavra (shape: n_subjects, n_channels, n_times)
X = np.stack(all_subjects['palavra']) - np.stack(all_subjects['pseudopalavra'])
print(f"Shape: {X.shape} (sujeitos, canais, tempos)")

# Rodar cluster permutation
t_obs, clusters, cluster_pv, H0 = spatio_temporal_cluster_1samp_test(
    X,
    n_permutations=1024,
    threshold=None,  # MNE calcula automaticamente (t baseado em p < .05)
    tail=0,  # two-sided
    seed=42,
)

# Resumir
n_sig = sum(cluster_pv < 0.05)
print(f"\nResultados:")
print(f"  Total de clusters encontrados: {len(clusters)}")
print(f"  Clusters significativos (p < .05): {n_sig}")
if n_sig > 0:
    print(f"  p do cluster mais significativo: {min(cluster_pv):.4f}")
    # Maior cluster
    largest = np.argmax([len(c[0]) for c in clusters])
    print(f"  Maior cluster: {len(clusters[largest][0])} eletrodos × {len(clusters[largest][1])} pontos temporais")

In [ ]:
# 5. Visualizar clusters significativos
print("\n=== Visualizando clusters significativos ===")

fig, ax = plt.subplots(figsize=(12, 6))

# t-map médio (média sobre tempos para visualização)
t_mean = t_obs.mean(axis=1)

# Plotar t-values por canal
bar_colors = ['red' if p < 0.05 else 'gray' for p in cluster_pv]
# Para simplicidade, plotar max(|t|) por canal
max_t_per_ch = np.abs(t_obs).max(axis=1)
    
ax.barh(range(n_channels), max_t_per_ch, color='steelblue', alpha=0.7)
ax.set_yticks(range(0, n_channels, 4))
ax.set_yticklabels([ch_names[i] for i in range(0, n_channels, 4)])
ax.set_xlabel('max(|t|) por canal')
ax.set_ylabel('Canal')
ax.set_title('Magnitude do efeito por canal (palavra vs. pseudopalavra)')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('11_cluster_tvalues.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Salvo: 11_cluster_tvalues.png")

In [ ]:
# 6. Análise por componente: extrair amplitudes e fazer GLM misto
print("\n=== Análise por componente (extrair amplitudes) ===\n")

# Para cada sujeito e condição, calcular amplitude média na janela
def get_amplitude(data, ch_indices, tmin, tmax, times):
    """Amplitude média na janela do componente."""
    mask = (times >= tmin) & (times <= tmax)
    return data[ch_indices][:, mask].mean() * 1e6  # µV

data_amp = []
for cond in ['palavra', 'pseudopalavra']:
    for subj_idx, subj_data in enumerate(all_subjects[cond]):
        for comp_name, config in ERP_COMPONENTS.items():
            chs = [ch for ch in config['channels'] if ch in ch_names]
            if not chs:
                continue
            ch_indices = [ch_names.index(ch) for ch in chs]
            tmin, tmax = config['time_range']
            amp = get_amplitude(subj_data, ch_indices, tmin, tmax, times)
            data_amp.append({
                'sujeito': f'subj_{subj_idx:02d}',
                'condicao': cond,
                'componente': comp_name,
                'amplitude': amp
            })

df_amp = pd.DataFrame(data_amp)
print(df_amp.head(10))
print(f"\n✅ DataFrame criado: {df_amp.shape[0]} observações")

In [ ]:
# 7. Teste t pareado por componente (palavra vs. pseudopalavra)
print("\n=== Teste t pareado por componente ===\n")

resultados = []
for comp in ERP_COMPONENTS.keys():
    df_comp = df_amp[df_amp['componente'] == comp]
    
    # T pareado (mesmo sujeito, duas condições)
    amp_palavra = df_comp[df_comp['condicao'] == 'palavra']['amplitude'].values
    amp_pseudo = df_comp[df_comp['condicao'] == 'pseudopalavra']['amplitude'].values
    
    t_stat, p_val = ttest_rel(amp_palavra, amp_pseudo)
    
    # Cohen's d para pareado
    diff = amp_palavra - amp_pseudo
    d = diff.mean() / diff.std()
    
    resultados.append({
        'Componente': comp,
        'Amp_palavra (M±DP)': f"{amp_palavra.mean():.2f} ± {amp_palavra.std():.2f}",
        'Amp_pseudo (M±DP)': f"{amp_pseudo.mean():.2f} ± {amp_pseudo.std():.2f}",
        't': round(t_stat, 3),
        'p': round(p_val, 4),
        'd_Cohen': round(d, 3),
        'Significativo': '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))
    })

df_resultados = pd.DataFrame(resultados)
print(df_resultados.to_string(index=False))
df_resultados.to_csv('12_componentes_test_t.csv', index=False)
print("\n✅ Salvo: 12_componentes_test_t.csv")

In [ ]:
# 8. Boxplot das amplitudes por componente
print("\n=== Visualização: amplitudes por componente ===")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, comp in zip(axes, ERP_COMPONENTS.keys()):
    df_comp = df_amp[df_amp['componente'] == comp]
    
    # Boxplot
    df_comp.boxplot(column='amplitude', by='condicao', ax=ax)
    ax.set_title(f'{comp}')
    ax.set_xlabel('Condição')
    ax.set_ylabel('Amplitude (µV)')
    plt.suptitle('')  # remove título padrão
    
    # Adicionar pontos individuais
    for i, cond in enumerate(['palavra', 'pseudopalavra']):
        y = df_comp[df_comp['condicao'] == cond]['amplitude'].values
        x = np.random.normal(i + 1, 0.04, size=len(y))
        ax.scatter(x, y, alpha=0.4, color=['#1f77b4', '#ff7f0e'][i], s=20)

fig.suptitle('Amplitude dos componentes ERP por condição', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('13_amplitude_boxplot.png', dpi=100, bbox_inches='tight')
plt.show()
print("✅ Salvo: 13_amplitude_boxplot.png")

In [ ]:
# 9. Resumo final
print("\n" + "=" * 50)
print("RESUMO DA ANÁLISE")
print("=" * 50)
print(f"\nAmostra: N={n_subjects} sujeitos")
print(f"Condições: palavra vs. pseudopalavra")
print(f"Componentes analisados: {list(ERP_COMPONENTS.keys())}")
print(f"\nTeste cluster-based permutation:")
print(f"  Clusters significativos: {n_sig} de {len(clusters)}")
print(f"\nTeste t pareado por componente:")
for _, row in df_resultados.iterrows():
    sig_marker = ' ✅' if row['Significativo'] != 'ns' else ' ❌'
    print(f"  {row['Componente']}: p = {row['p']}, d = {row['d_Cohen']}{sig_marker}")

print("\n✅ Análise completa. Figuras e tabelas salvas.")
print("\nPróximos passos:")
print("  - Carregar dados REAIS do P03")
print("  - Aumentar N (poder estatístico)")
print("  - Adicionar covariáveis (idade, sexo, SES)")
print("  - Integrar com modelo linear misto (statsmodels)")

# Conclusões

## Quando usar cada teste

| Teste | Quando usar | Vantagens |
|---|---|---|
| **Cluster permutation** | Comparar duas condições em todo o espaço-tempo | Controla múltiplas comparações; não assume a priori quais eletrodos/tempos |
| **Teste t por componente** | Quando há hipótese específica sobre componente (N170, N400) | Mais simples, mais interpretável |
| **GLM misto** | Quando há covariáveis e medidas repetidas | Modelo mais flexível |
| **Bootstrap** | Para ICs de qualquer estatística | Não-paramétrico |

## Referências\n
- **Maris & Oostenveld (2007).** Nonparametric statistical testing of EEG- and MEG-data. *Journal of Neuroscience Methods*.
- **Luck (2014).** An Introduction to the Event-Related Potential Technique.
- **Cohen (2014).** Analyzing Neural Time Series Data.

## Recursos\n
- [MNE: Statistical analysis](https://mne.tools/mne-1.5/auto_tutorials/stats-sensor-space/index.html)
- [Permutation tests for ERP analysis](https://doi.org/10.1016/j.jneumeth.2007.03.024)